In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV


In [2]:
import os

# Ver en qué carpeta está ejecutándose el notebook
print("Carpeta actual:", os.getcwd())

# Ver qué hay ahí
for archivo in os.listdir("."):
    print(archivo)

Carpeta actual: /mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/notebooks
.ipynb_checkpoints
analysis1.ipynb
catboost_info
compute_distance_in_parallel.ipynb
Step2_IncCoordinates.ipynb
Step2_Merge.ipynb
Step3 Modelling.ipynb


In [3]:
y_train = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/y_train.csv").head(3)

In [4]:
#load the data
X_train = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/X_train_final.csv")
X_test  = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/X_test_final.csv")
y_train = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/y_train.csv").squeeze()
y_test  = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/y_test.csv").squeeze()
y_test_log =pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/y_test_log.csv").squeeze()
y_train_log = pd.read_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/y_train_log.csv").squeeze()
#squeeze()because we have only 1Dimension serie and to avoida the warnings with scikit-learns
#in ubuntu e: it´s /mnt/e

In [5]:
def evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    return {"Model": nombre, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2 Score": round(r2, 4)}

# Baseline (predecir siempre la media)
y_pred_naive = np.full(len(y_test), y_train.mean())
naive_result = {
    "Model": "Naive Baseline (Mean)",
    "MAE":   round(mean_absolute_error(y_test, y_pred_naive), 2),
    "RMSE":  round(np.sqrt(mean_squared_error(y_test, y_pred_naive)), 2),
    "R2 Score": round(r2_score(y_test, y_pred_naive), 4)
}

# Modelos
modelos = [
    ("Random Forest (Simple)", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ("Linear Regression",      LinearRegression()),
    ("KNN Regressor",          KNeighborsRegressor(n_neighbors=5, n_jobs=-1)),
]

resultados = [evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test)
              for nombre, modelo in modelos]
resultados.append(naive_result)

#Tabla de resultados
df_resultados = pd.DataFrame(resultados).sort_values("R2 Score", ascending=False)
print(df_resultados.to_string(index=False))

                 Model   MAE   RMSE  R2 Score
Random Forest (Simple) 52.60  72.88    0.5580
     Linear Regression 56.71  76.71    0.5104
         KNN Regressor 69.18  90.68    0.3158
 Naive Baseline (Mean) 85.94 109.98   -0.0065


In [6]:
#reduce drastically hte grid, because it dies mutliple times
param_grids = {
    "Linear Regression": {
        'model':  LinearRegression(),
        'params': {},
        'use_log': True
    },
    "KNN": {
        'model':  KNeighborsRegressor(),
        'params': {
            'n_neighbors': [15, 19, 25],   # solo 3 valores, ya sabemos que 19 funciona bien
            'weights':     ['distance']     # ya sabemos que distance gana
        },
        'use_log': True
    },
    "Random Forest": {
        'model':  RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            max_samples=0.3    # usa solo el 30% de filas en cada árbol → mucho menos RAM
        ),
        'params': {
            'n_estimators': [100],          # solo 1 valor
            'max_depth':    [10, 20],       # quitar None (árbol completo = enorme)
            'min_samples_split': [5]        # solo 1 valor
        },
        'use_log': False
    }
}

results = []

for name, cfg in param_grids.items():
    print(f"\nModel: {name}:")

    if cfg['use_log']:
        y_tr = y_train_log
    else:
        y_tr = y_train

    grid = GridSearchCV(cfg['model'], cfg['params'], cv=5, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_tr)        # <-- X_train en lugar de X_train_filtered

    print(f"  Better parameters: {grid.best_params_ if grid.best_params_ else '---'}")
    print(f"  CV R² medio: {grid.best_score_:.4f}")

    y_pred = grid.predict(X_test)  # <-- X_test en lugar de X_test_filtered

    if cfg['use_log']:
        y_pred = np.exp(y_pred)

    results.append({
        'Model': name,
        'R²':    r2_score(y_test, y_pred),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'MSE':   mean_squared_error(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred))
    })

results_df = pd.DataFrame(results).sort_values('R²', ascending=False).round(4)
print("\nResumen de rendimiento:")
print(results_df.to_string(index=False))


Model: Linear Regression:
  Better parameters: ---
  CV R² medio: 0.4959

Model: KNN:
  Better parameters: {'n_neighbors': 25, 'weights': 'distance'}
  CV R² medio: 0.3897

Model: Random Forest:
  Better parameters: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100}
  CV R² medio: 0.5623

Resumen de rendimiento:
            Model     R²     MAE       MSE    RMSE
    Random Forest 0.5606 52.4004 5281.0435 72.6708
Linear Regression 0.3953 59.0521 7267.0371 85.2469
              KNN 0.3550 66.1105 7751.4143 88.0421


In [7]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

modelos_avanzados = {
    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
        tree_method='hist'
    ),
    "LightGBM": LGBMRegressor(
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    "CatBoost": CatBoostRegressor(
        random_seed=42,
        verbose=0
    )
}

results_avanzados = []

for name, model in modelos_avanzados.items():
    print(f"Entrenando {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results_avanzados.append({
        'Model': name,
        'R²':   r2_score(y_test, y_pred),
        'MAE':  mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MSE':   mean_squared_error(y_test, y_pred),
    })
    print(f"  R²: {results_avanzados[-1]['R²']:.4f} | MAE: {results_avanzados[-1]['MAE']:.4f} | RMSE: {results_avanzados[-1]['RMSE']:.4f}")
    

Entrenando XGBoost
  R²: 0.5760 | MAE: 51.0389 | RMSE: 71.3864
Entrenando LightGBM
  R²: 0.5643 | MAE: 52.0868 | RMSE: 72.3648
Entrenando CatBoost
  R²: 0.5793 | MAE: 50.7938 | RMSE: 71.1073


In [8]:
# Comparativa con modelos anteriores

df_comparativa = pd.concat([
    pd.DataFrame(results_avanzados),
    results_df
]).sort_values('R²', ascending=False).round(4)

print("\nComparativa completa:")
print(df_comparativa.to_string(index=False)) 


Comparativa completa:
            Model     R²     MAE    RMSE       MSE
         CatBoost 0.5793 50.7938 71.1073 5056.2511
          XGBoost 0.5760 51.0389 71.3864 5096.0220
         LightGBM 0.5643 52.0868 72.3648 5236.6602
    Random Forest 0.5606 52.4004 72.6708 5281.0435
Linear Regression 0.3953 59.0521 85.2469 7267.0371
              KNN 0.3550 66.1105 88.0421 7751.4143


In [9]:
#XGBoost GridSearch
print("Entrenando XGBoost con GridSearch:")

grid_xgb = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'),
    {
        'n_estimators':  [100, 200],
        'max_depth':     [6, 10],
        'learning_rate': [0.05, 0.1]
    },
    cv=5, scoring='r2', n_jobs=-1
)
grid_xgb.fit(X_train, y_train)
y_pred_xgb = grid_xgb.predict(X_test)

result_xgb_grid = {
    'Model': 'XGBoost (GridSearch)',
    'R²':    r2_score(y_test, y_pred_xgb),
    'MAE':   mean_absolute_error(y_test, y_pred_xgb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_xgb))
}
print(f"  Better parameters: {grid_xgb.best_params_}")
print(f"  CV R² medio: {grid_xgb.best_score_:.4f}")
print(f"  Test R²: {result_xgb_grid['R²']:.4f} | MAE: {result_xgb_grid['MAE']:.4f} | RMSE: {result_xgb_grid['RMSE']:.4f}")

Entrenando XGBoost con GridSearch:
  Better parameters: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200}
  CV R² medio: 0.5810
  Test R²: 0.5818 | MAE: 50.4795 | RMSE: 70.8923


In [11]:
# LightGBM GridSearch 
print("Entrenando LightGBM con GridSearch...")

grid_lgbm = GridSearchCV(
     LGBMRegressor(
        random_state=42,
        n_jobs=1, #giving the use of all the nucelos of the CPU and in Parallel with Grdisearch (change to 4)
        verbose=-1,
        boosting_type='gbdt',    # más estable y rápido que dart
        num_leaves=31,           # limita el crecimiento por hojas
        subsample=0.8,           # usa 80% de filas por árbol
        colsample_bytree=0.8     # usa 80% de features por árbol
    ),
    {
        'n_estimators':  [100, 200],
        'max_depth':     [6, 10],
        'learning_rate': [0.05, 0.1]
    },
    cv=5, scoring='r2', n_jobs=4
)
grid_lgbm.fit(X_train, y_train)
y_pred_lgbm = grid_lgbm.predict(X_test)

result_lgbm_grid = {
    'Model': 'LightGBM (GridSearch)',
    'R²':    r2_score(y_test, y_pred_lgbm),
    'MAE':   mean_absolute_error(y_test, y_pred_lgbm),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
}
print(f"  Better parameters: {grid_lgbm.best_params_}")
print(f"  CV R² medio: {grid_lgbm.best_score_:.4f}")
print(f"  Test R²: {result_lgbm_grid['R²']:.4f} | MAE: {result_lgbm_grid['MAE']:.4f} | RMSE: {result_lgbm_grid['RMSE']:.4f}")

Entrenando LightGBM con GridSearch...
  Better parameters: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200}
  CV R² medio: 0.5739
  Test R²: 0.5720 | MAE: 51.4349 | RMSE: 71.7210


In [14]:
# CatBoost GridSearch
print("Entrenando CatBoost con GridSearch...")

grid_cb = GridSearchCV(
    CatBoostRegressor(random_seed=42, verbose=0),
    {
        'iterations':    [100, 200],
        'depth':         [6, 10],
        'learning_rate': [0.05, 0.1]
    },
    cv=5, scoring='r2', n_jobs=-1
)
grid_cb.fit(X_train, y_train)
y_pred_cb = grid_cb.predict(X_test)

result_cb_grid = {
    'Model': 'CatBoost (GridSearch)',
    'R²':    r2_score(y_test, y_pred_cb),
    'MAE':   mean_absolute_error(y_test, y_pred_cb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_cb))
}
print(f"  Better parameters: {grid_cb.best_params_}")
print(f"  CV R² medio: {grid_cb.best_score_:.4f}")
print(f"  Test R²: {result_cb_grid['R²']:.4f} | MAE: {result_cb_grid['MAE']:.4f} | RMSE: {result_cb_grid['RMSE']:.4f}")

Entrenando CatBoost con GridSearch...
  Better parameters: {'depth': 10, 'iterations': 200, 'learning_rate': 0.1}
  CV R² medio: 0.5739
  Test R²: 0.5725 | MAE: 51.3931 | RMSE: 71.6758


In [16]:
#compare of old results and new ones

df_comparativa2 = pd.concat([
    df_comparativa,
    pd.DataFrame([result_xgb_grid, result_lgbm_grid, result_cb_grid])
]).sort_values('R²', ascending=False).round(4)

print("\nComplete comparative:")
print(df_comparativa2.to_string(index=False))


Complete comparative:
                Model     R²     MAE    RMSE       MSE
 XGBoost (GridSearch) 0.5818 50.4795 70.8923       NaN
             CatBoost 0.5793 50.7938 71.1073 5056.2511
              XGBoost 0.5760 51.0389 71.3864 5096.0220
CatBoost (GridSearch) 0.5725 51.3931 71.6758       NaN
LightGBM (GridSearch) 0.5720 51.4349 71.7210       NaN
             LightGBM 0.5643 52.0868 72.3648 5236.6602
        Random Forest 0.5606 52.4004 72.6708 5281.0435
    Linear Regression 0.3953 59.0521 85.2469 7267.0371
                  KNN 0.3550 66.1105 88.0421 7751.4143


In [17]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

#XGBoost RandomizedSearch
print("Training XGBoost with RandomizedSearchCV...")

xgb_params = {
    'n_estimators':  randint(100, 400),
    'max_depth':     randint(4, 12),
    'learning_rate': uniform(0.01, 0.2),
    'subsample':     uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

random_xgb = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'),
    param_distributions=xgb_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1, 
    random_state=42
)
random_xgb.fit(X_train, y_train)
y_pred_rxgb = random_xgb.predict(X_test)

result_xgb_random = {
    'Model': 'XGBoost (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rxgb),
    'MAE':   mean_absolute_error(y_test, y_pred_rxgb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rxgb))
}
print(f"  Best parameters: {random_xgb.best_params_}")
print(f"  CV R² mean: {random_xgb.best_score_:.4f}")
print(f"  Test R²: {result_xgb_random['R²']:.4f} | MAE: {result_xgb_random['MAE']:.4f} | RMSE: {result_xgb_random['RMSE']:.4f}")

Training XGBoost with RandomizedSearchCV...
  Best parameters: {'colsample_bytree': np.float64(0.9140703845572055), 'learning_rate': np.float64(0.04993475643167195), 'max_depth': 10, 'n_estimators': 343, 'subsample': np.float64(0.836965827544817)}
  CV R² mean: 0.5823
  Test R²: 0.5845 | MAE: 50.3077 | RMSE: 70.6611


In [20]:
#LightGBM RandomizedSearch
print("Training LightGBM with RandomizedSearchCV...")

lgbm_params = {
    'n_estimators':     randint(100, 400),
    'max_depth':        randint(4, 12),
    'learning_rate':    uniform(0.01, 0.2),
    'num_leaves':       randint(20, 100),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

random_lgbm = RandomizedSearchCV(
    LGBMRegressor(random_state=42, n_jobs=2, verbose=-1),
    param_distributions=lgbm_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=4,
    random_state=42
)
random_lgbm.fit(X_train, y_train)
y_pred_rlgbm = random_lgbm.predict(X_test)

result_lgbm_random = {
    'Model': 'LightGBM (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rlgbm),
    'MAE':   mean_absolute_error(y_test, y_pred_rlgbm),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rlgbm))
}
print(f"  Best parameters: {random_lgbm.best_params_}")
print(f"  CV R² mean: {random_lgbm.best_score_:.4f}")
print(f"  Test R²: {result_lgbm_random['R²']:.4f} | MAE: {result_lgbm_random['MAE']:.4f} | RMSE: {result_lgbm_random['RMSE']:.4f}")

Training LightGBM with RandomizedSearchCV...
  Best parameters: {'colsample_bytree': np.float64(0.9579309401710595), 'learning_rate': np.float64(0.12957999576221704), 'max_depth': 11, 'n_estimators': 370, 'num_leaves': 91, 'subsample': np.float64(0.8281775897621597)}
  CV R² mean: 0.5815
  Test R²: 0.5843 | MAE: 50.3130 | RMSE: 70.6842


In [21]:
# ── CatBoost RandomizedSearch ──────────────────────────────────
print("Training CatBoost with RandomizedSearchCV...")

cb_params = {
    'iterations':    randint(100, 400),
    'depth':         randint(4, 10),
    'learning_rate': uniform(0.01, 0.2),
    'l2_leaf_reg':   randint(1, 10)
}

random_cb = RandomizedSearchCV(
    CatBoostRegressor(random_seed=42, verbose=0),
    param_distributions=cb_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)
random_cb.fit(X_train, y_train)
y_pred_rcb = random_cb.predict(X_test)

result_cb_random = {
    'Model': 'CatBoost (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rcb),
    'MAE':   mean_absolute_error(y_test, y_pred_rcb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rcb))
}
print(f"  Best parameters: {random_cb.best_params_}")
print(f"  CV R² mean: {random_cb.best_score_:.4f}")
print(f"  Test R²: {result_cb_random['R²']:.4f} | MAE: {result_cb_random['MAE']:.4f} | RMSE: {result_cb_random['RMSE']:.4f}")

Training CatBoost with RandomizedSearchCV...
  Best parameters: {'depth': 7, 'iterations': 363, 'l2_leaf_reg': 3, 'learning_rate': np.float64(0.19186408041575642)}
  CV R² mean: 0.5779
  Test R²: 0.5781 | MAE: 50.8827 | RMSE: 71.2067


In [31]:
df_final = pd.concat([
    pd.DataFrame([result_xgb_random, result_lgbm_random, result_cb_random]),
    df_comparativa2
    ]).sort_values('R²', ascending=False).round(4).reset_index(drop=True)

df_final = df_final.drop(df_final.columns[4], axis = 1)

print("\nFull model comparison:")
print(df_final.to_string(index=False))


Full model comparison:
                      Model     R²     MAE    RMSE
 XGBoost (RandomizedSearch) 0.5845 50.3077 70.6611
LightGBM (RandomizedSearch) 0.5843 50.3130 70.6842
       XGBoost (GridSearch) 0.5818 50.4795 70.8923
                   CatBoost 0.5793 50.7938 71.1073
CatBoost (RandomizedSearch) 0.5781 50.8827 71.2067
                    XGBoost 0.5760 51.0389 71.3864
      CatBoost (GridSearch) 0.5725 51.3931 71.6758
      LightGBM (GridSearch) 0.5720 51.4349 71.7210
                   LightGBM 0.5643 52.0868 72.3648
              Random Forest 0.5606 52.4004 72.6708
          Linear Regression 0.3953 59.0521 85.2469
                        KNN 0.3550 66.1105 88.0421


In [32]:
#we continue in the next notebook, so we save the results and the best model using joblib

# Save all results to CSV
df_final.to_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/model_comparison_results.csv", index=False)


#Save the best model

import joblib

joblib.dump(random_xgb, "/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/best_model.pkl")


['/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/best_model.pkl']